In [1]:
# Exercise 8 : using FAISS for eficient search of vectors

from pypdf import PdfReader

reader = PdfReader("os.pdf")
text = ""
for page in reader.pages:
    text += page.extract_text() + "\n"


In [2]:
def chunk_text(text, chunk_size=100, overlap=20):

    words = text.split()
    step = chunk_size - overlap
    chunks = []
    for i in range(0,len(words),step):
        chunk = " ".join(words[i:i+chunk_size])

        if chunk:
            chunks.append(chunk)
    return chunks

chunks = chunk_text(text)

In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

chunk_embeddings = model.encode(chunks)

W0629 17:20:00.235000 26496 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
C:\Users\uniqu\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [4]:
import numpy as np
chunk_embeddings = np.array(
    chunk_embeddings,
    dtype=np.float32
)

In [5]:
import faiss
dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)

index.add(chunk_embeddings)

In [12]:
def retrieve(query, k=3):
    query_embedding = model.encode([query])

    query_embedding = np.array(
        query_embedding,
        dtype=np.float32
    )

    distances, indices = index.search(
        query_embedding,
        k
    )

    return [chunks[i] for i in indices[0]]

In [7]:
def build_prompt(query, retrieved_chunks):
    context = "\n".join(retrieved_chunks)
    prompt = f"""
Context :
{context}

Question :
{query}

Answer using this above context only

"""
    return prompt


In [ ]:
from google import genai

client = genai.Client(api_key="GEMINI_API_KEY")

while True:

    query = input("Ask: ")

    if query == "exit":
        break

    retrieved_chunks = retrieve(query,3)

    prompt = build_prompt(
        query,
        retrieved_chunks
    )
    # print(retrieved_chunks)

    response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
    )

    print("Answer : ")
    print(response.text)



Ask:  what is paging?


Answer : 
Based on the context, paging:
*   Causes a little internal fragmentation in the last page.
*   Uses an addressing scheme where the address is just a page number plus offset, which has no real meaning to the programmer.


Ask:  exit
